# ConvoBridge — Qwen2.5-0.5B Meeting Chatbot (Colab)

Lighter alternative to **Gemma-3-1B** for transcript Q&A.

| Item | Value |
|---|---|
| Base model | `Qwen/Qwen2.5-0.5B-Instruct` |
| Method | LoRA (Causal LM) |
| Data | SamSum + SQuAD (online) |
| Output | Google Drive `ConvoBridge/qwen25-05b-chatbot-lora/` |
| CPU RAM (infer) | ~1.5–3 GB |

**Runtime → Change runtime type → GPU (T4)**

Separate from Gemma chatbot notebook.


## Step 1 — Check GPU


In [ ]:
!nvidia-smi


## Step 2 — Install dependencies


In [ ]:
!pip install -q "transformers>=4.44.0" "peft>=0.13.0" "datasets>=2.20.0" \
  "accelerate>=0.33.0" "trl>=0.9.0" bitsandbytes huggingface_hub pyyaml


## Step 3 — Mount Google Drive


In [ ]:
from pathlib import Path
from google.colab import drive

drive.mount("/content/drive")
DRIVE_ROOT = Path("/content/drive/MyDrive/ConvoBridge")
OUTPUT_DIR = DRIVE_ROOT / "qwen25-05b-chatbot-lora"
CHECKPOINT_DIR = DRIVE_ROOT / "qwen05_chatbot_checkpoints"
DRIVE_ROOT.mkdir(parents=True, exist_ok=True)
CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)
print("Drive ready:", DRIVE_ROOT)


## Step 4 — Config


In [ ]:
from pathlib import Path
import torch

assert torch.cuda.is_available(), "Enable GPU: Runtime -> Change runtime type -> GPU"
torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.allow_tf32 = True
torch.cuda.empty_cache()

print("GPU:", torch.cuda.get_device_name(0))
print("VRAM GB:", round(torch.cuda.get_device_properties(0).total_memory / 1e9, 2))

BASE_MODEL = "Qwen/Qwen2.5-0.5B-Instruct"

PROMPT_TEMPLATE = """You are ConvoBridge Meeting Q&A Assistant.
Answer the user question using ONLY the meeting transcript context below.

Rules:
- Use only facts present in the transcript.
- If the answer is not in the transcript, reply exactly: Not mentioned in the transcript.
- Keep the answer short and clear (1-4 sentences).
- Keep the same language as the question when possible.

TRANSCRIPT:
{transcript}

QUESTION:
{question}
"""

TRAIN_CFG = {
    "use_qlora": False,  # set True if OOM
    "lora_r": 16,
    "lora_alpha": 32,
    "lora_dropout": 0.05,
    "learning_rate": 2e-4,
    "num_epochs": 2,
    "per_device_train_batch_size": 8,
    "gradient_accumulation_steps": 2,
    "max_seq_length": 1024,
    "save_steps": 100,
    "max_train_rows": 6000,
    "gradient_checkpointing": True,
}

QUESTION_BANK = [
    "What was discussed?",
    "What are the main points?",
    "What decisions were made?",
    "What are the next steps?",
    "Who is responsible for follow-up?",
    "When is the next meeting?",
    "What problems were mentioned?",
    "Summarize the conversation briefly.",
]

print("BASE_MODEL:", BASE_MODEL)
print("OUTPUT_DIR:", OUTPUT_DIR)


## Step 5 — Build Q&A training rows (SamSum + SQuAD)


In [ ]:
import random
from datasets import Dataset, load_dataset

random.seed(42)

def clip(text, max_chars=1800):
    text = (text or "").strip()
    return text if len(text) <= max_chars else text[: max_chars - 3] + "..."

max_rows = TRAIN_CFG["max_train_rows"]
samsum_n = int(max_rows * 0.7)
squad_n = max_rows - samsum_n

print("Loading SamSum...")
samsum = load_dataset("knkarthick/samsum", split="train")
rows = []
for ex in samsum:
    dialogue = (ex.get("dialogue") or "").strip()
    summary = (ex.get("summary") or "").strip()
    if len(dialogue) < 40 or len(summary) < 10:
        continue
    rows.append({
        "transcript": dialogue,
        "question": random.choice(QUESTION_BANK),
        "answer": summary,
    })
    if len(rows) >= samsum_n:
        break

print("Loading SQuAD...")
squad = load_dataset("rajpurkar/squad", split="train")
for ex in squad:
    context = (ex.get("context") or "").strip()
    question = (ex.get("question") or "").strip()
    answers = ex.get("answers") or {}
    texts = answers.get("text") or []
    if not context or not question or not texts:
        continue
    answer = texts[0].strip()
    if not answer:
        continue
    rows.append({"transcript": context, "question": question, "answer": answer})
    if len(rows) % 5 == 0:
        rows.append({
            "transcript": context,
            "question": "What is the company's IPO date?",
            "answer": "Not mentioned in the transcript.",
        })
    if len(rows) >= max_rows:
        break

random.shuffle(rows)
print("Total rows:", len(rows))
print("Sample Q:", rows[0]["question"])
print("Sample A:", rows[0]["answer"][:120])


## Step 6 — Train Qwen2.5-0.5B + LoRA


In [ ]:
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig, TrainingArguments
from trl import SFTTrainer

try:
    from trl import SFTConfig
except ImportError:
    SFTConfig = None

tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

if TRAIN_CFG["use_qlora"]:
    bnb = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_compute_dtype=torch.bfloat16,
        bnb_4bit_use_double_quant=True,
    )
    model = AutoModelForCausalLM.from_pretrained(
        BASE_MODEL,
        quantization_config=bnb,
        device_map="auto",
        trust_remote_code=True,
    )
    model = prepare_model_for_kbit_training(model)
else:
    model = AutoModelForCausalLM.from_pretrained(
        BASE_MODEL,
        torch_dtype=torch.bfloat16,
        device_map="auto",
        trust_remote_code=True,
    )

lora = LoraConfig(
    r=TRAIN_CFG["lora_r"],
    lora_alpha=TRAIN_CFG["lora_alpha"],
    lora_dropout=TRAIN_CFG["lora_dropout"],
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
)
model = get_peft_model(model, lora)
model.print_trainable_parameters()

def to_text(row):
    user = PROMPT_TEMPLATE.format(
        transcript=clip(row["transcript"]),
        question=row["question"].strip(),
    )
    messages = [
        {"role": "user", "content": user},
        {"role": "assistant", "content": row["answer"]},
    ]
    return tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=False
    )

texts = [to_text(r) for r in rows]
ds = Dataset.from_dict({"text": texts})

common = dict(
    output_dir=str(CHECKPOINT_DIR),
    num_train_epochs=TRAIN_CFG["num_epochs"],
    per_device_train_batch_size=TRAIN_CFG["per_device_train_batch_size"],
    gradient_accumulation_steps=TRAIN_CFG["gradient_accumulation_steps"],
    learning_rate=TRAIN_CFG["learning_rate"],
    logging_steps=20,
    save_steps=TRAIN_CFG["save_steps"],
    save_total_limit=2,
    bf16=True,
    gradient_checkpointing=TRAIN_CFG["gradient_checkpointing"],
    report_to="none",
)

max_len = TRAIN_CFG["max_seq_length"]
if SFTConfig is not None:
    try:
        args = SFTConfig(max_length=max_len, **common)
        trainer = SFTTrainer(
            model=model,
            args=args,
            train_dataset=ds,
            processing_class=tokenizer,
        )
    except TypeError:
        args = TrainingArguments(**common)
        trainer = SFTTrainer(
            model=model,
            args=args,
            train_dataset=ds,
            tokenizer=tokenizer,
            max_seq_length=max_len,
            dataset_text_field="text",
        )
else:
    args = TrainingArguments(**common)
    trainer = SFTTrainer(
        model=model,
        args=args,
        train_dataset=ds,
        tokenizer=tokenizer,
        max_seq_length=max_len,
        dataset_text_field="text",
    )

trainer.train()
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
trainer.model.save_pretrained(str(OUTPUT_DIR))
tokenizer.save_pretrained(str(OUTPUT_DIR))
print("Saved ->", OUTPUT_DIR)
print("Files:", sorted(p.name for p in OUTPUT_DIR.iterdir()))


## Step 7 — Quick inference test


In [ ]:
from peft import PeftModel

sample_transcript = """Alice: We need to finish the backend by Friday.
Bob: I will handle the deployment.
Alice: Great, let's sync Monday at 10 AM. Sara will handle design."""

tok = AutoTokenizer.from_pretrained(BASE_MODEL, trust_remote_code=True)
if tok.pad_token is None:
    tok.pad_token = tok.eos_token

base = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL, torch_dtype=torch.bfloat16, device_map="auto", trust_remote_code=True
)
mdl = PeftModel.from_pretrained(base, str(OUTPUT_DIR))
mdl.eval()

question = "Who will handle design?"
user = PROMPT_TEMPLATE.format(transcript=sample_transcript, question=question)
messages = [{"role": "user", "content": user}]
text = tok.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
inputs = tok(text, return_tensors="pt").to(mdl.device)
with torch.no_grad():
    out = mdl.generate(**inputs, max_new_tokens=128, do_sample=False, pad_token_id=tok.pad_token_id)
gen = out[0][inputs["input_ids"].shape[-1]:]
print(tok.decode(gen, skip_special_tokens=True).strip())


## Step 8 — Download to your laptop

1. Google Drive → `MyDrive/ConvoBridge/qwen25-05b-chatbot-lora/`
2. Download the folder
3. Copy into your project:

```text
chatbot/models/qwen25-05b-chatbot-lora/
```

If CUDA OOM during training, set in Step 4:

```python
TRAIN_CFG["use_qlora"] = True
TRAIN_CFG["per_device_train_batch_size"] = 4
```

Backend wiring to this model is a separate step (ask when ready).
